Import potrzebnych bibliotek.

In [ ]:

import os
import pandas as pd
import pickle
import re
import pdfplumber
import numpy as np


Przetwarzanie pobranych plików pdf ze stenogramami i zapisywanie ich do pliku pickle. Strona w stenogramie jest zapisana w formacie dwukolumnowym - postarano się jak najlepiej umożliwić odczytanie takich plików. (Jest zakomentowane, aby przypadkowo nie uruchomić - przetwarzanie folderu z jednej kadencji zajmowało ok. 3 godziny).

In [ ]:
'''
#konfiguracja ścieżek i folder wyjściowy
INPUT_ROOT = "stenograms"
OUTPUT_ROOT = "data_new"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

#foldery do przetworzenia (przetwarzałem kilkukrotnie)
#folders_to_process = ["8","9","10"] 
folders_to_process = ["10"] 

for folder in folders_to_process:
    print(f"\n=== Processing folder {folder} ===")
    
    output_path = os.path.join(OUTPUT_ROOT, f"texts_{folder}.pkl")
    folder_path = os.path.join(INPUT_ROOT, folder)
    
    with open(output_path, "ab") as output:
        #licznik bo chciałem przerwać po kilku plikach żeby sprawdzić jak działa przy testowaniu
        i = 0
        
        #sortowanie plików
        files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")])

        for file in files:
            #sprawdzenie dla 1 pliku
            #if i == 1:
            #    break
            if i %100 == 0:
                print(f"Przeprocesowano {i} plikow")
                
            try:
                pdf_path = os.path.join(folder_path, file)
                print(f"Processing: {file}")

                doc_content = ""
                #otwieranie pdf
                with pdfplumber.open(pdf_path) as pdf:
                    for page in pdf.pages:
                        width = page.width
                        height = page.height
                        #dzielenie na dwie kolumny
                        left_bbox = (0, 0, width / 2, height)
                        right_bbox = (width / 2, 0, width, height)
                        #ekstraktowanie tekstów z odpowiednich kolumn
                        left_text = page.within_bbox(left_bbox).extract_text()
                        right_text = page.within_bbox(right_bbox).extract_text()

                        #sklejenie w całość
                        if left_text:
                            doc_content += left_text + "\n"
                        if right_text:
                            doc_content += right_text + "\n"

                #wstępne czyszczenie tekstu 
                doc_content = re.sub(
                    r"[^A-Za-z0-9ąćęłńóśźżĄĆĘŁŃÓŚŹŻ.?!,-:()\n\r]+",
                    " ",
                    doc_content
                )
                doc_content = re.sub(r'\s+', ' ', doc_content).strip()
                #zapisanie do pickle
                pickle.dump(
                    {
                        "file": file,
                        "text": doc_content
                    },
                    output,
                    protocol=pickle.HIGHEST_PROTOCOL
                )

                i += 1

            except Exception as e:
                print(f"Error processing {file}: {e}")
                continue
'''


=== Processing folder 10 ===
Przeprocesowano 0 plikow
Processing: posiedzenie_10_2024-04-24.pdf
Processing: posiedzenie_10_2024-04-25.pdf
Processing: posiedzenie_10_2024-04-26.pdf
Processing: posiedzenie_11_2024-05-08.pdf
Processing: posiedzenie_11_2024-05-09.pdf
Processing: posiedzenie_11_2024-05-15.pdf
Processing: posiedzenie_12_2024-05-22.pdf
Processing: posiedzenie_12_2024-05-23.pdf
Processing: posiedzenie_13_2024-06-12.pdf
Processing: posiedzenie_13_2024-06-13.pdf
Processing: posiedzenie_13_2024-06-14.pdf
Processing: posiedzenie_14_2024-06-26.pdf
Processing: posiedzenie_14_2024-06-27.pdf
Processing: posiedzenie_14_2024-06-28.pdf
Processing: posiedzenie_15_2024-07-11.pdf
Processing: posiedzenie_15_2024-07-12.pdf
Processing: posiedzenie_16_2024-07-23.pdf
Processing: posiedzenie_16_2024-07-24.pdf
Processing: posiedzenie_16_2024-07-25.pdf
Processing: posiedzenie_16_2024-07-26.pdf
Processing: posiedzenie_17_2024-09-11.pdf
Processing: posiedzenie_17_2024-09-12.pdf
Processing: posiedzen

Wczytanie z formatu pickle do listy. 

In [ ]:
def load_pickle_stream(path):
    with open(path, "rb") as f:
        while True:
            try:
                yield pickle.load(f)
            except EOFError:
                break

path = os.path.join("data_new", "texts_8.pkl")
texts = (doc["text"] for doc in load_pickle_stream(path))


Wstępne czyszczenie - usunięcie znaków poza literami łacińskimi, polskimi oraz kilkoma znakami interpunkcyjnymi oraz znakami newline oraz zapisanie do pandas.dataframe. (Zainspirowane notatnikami)

In [ ]:
texts = pd.Series(texts).map(
    lambda x: re.sub(
        r"[^A-Za-z0-9ąćęłńóśźżĄĆĘŁŃÓŚŹŻ.?!,-:()\n\r]+",
        " ",
        x
    )
)


In [28]:
print(type(texts), np.shape(texts))

<class 'pandas.core.series.Series'> (235,)


Funkcja czyszcząca stenogram. Jej działanie to kolejno:
- znalezienie i usunięcie wszystkiego aż do wypowiedzi Marszałka/Wicemarszałka wznawiajacego obrady,
- usunięcie ciągów kropek (po spisie treści z pierwszych stron),
- usunięcie stopki wydawniczej z końca pliku,
- usunięcie wyrażeń w okrągłych nawiasach wraz z zawartością (są to wypowiedzi z sali w trakcie wystąpienia mówcy na mównicy i zawierają się w nich takie rzeczy jak (oklaski, gwizdy) lub cytaty posłów z sali)
- usunięcie "Spis treści" ale pisanego od końca (na co drugiej stronie w pliku pdf występował odnośnik do spisu treści, a po wczytaniu pliku było to zapisane od końca)

In [ ]:
def clean_sejm_stenogram(text: str) -> str:
    if not isinstance(text, str):
        return text

    original = text

    #ucina wszystko do pierwszej wypowiedzi marszałka
    match = re.search(
        r"(Otwieram|Wznawiam)\s+posiedzenie\b.*?[\.!?]",
        text,
        flags=re.IGNORECASE
    )
    if match:
        text = text[match.start():]
    else:
        text = original 

    #ciągi kropek
    text = re.sub(
        r"(?:\s*\.\s*){6,}",
        " ",
        text
    )

    #stopka wydawnicza
    text = re.sub(
        r"TŁOCZONO POLECENIA MARSZAŁKA SEJMU.*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    #nawiasy okrągłe - przerywniki w wypowiedzi
    text = re.sub(r"\([^)]*\)", " ", text)

    #naprawa rozbitych wyrazów
    text = re.sub(r"([A-Za-ząćęłńóśźżĄĆĘŁŃÓŚŹŻ])-\s+([A-Za-ząćęłńóśźżĄĆĘŁŃÓŚŹŻ])",r"\1\2",text)

    #nadmiarowe spacje
    text = re.sub(r'\s+', ' ', text).strip()

    #błędny spis treści 
    text = re.sub(r'icśert sipS', '', text)


    return text

texts = texts.map(clean_sejm_stenogram)

Usunięcie liczb, jednowyrazowych słów, spacji. (Inspirowane notebookiem dot. alimentów (możliwe, że nawet stamtąd przekopiowane))

In [ ]:
# usunięcie wszystkich cyfr i liczb 
texts = texts.map(lambda x: re.sub(r'[0-9]+', '', x))

# usunięcie jednoliterowych wyrazów
texts = texts.map(lambda x: re.sub(r'\b\w\b', '', x))

# usunięcie nadmiarowych spacji
texts = texts.map(lambda x: re.sub(r'\s+', ' ', x).strip())

Funkcja do anotowania roli mówcy zgodnie z kluczem ze słownika. Wyszukuje wyrażenia, w których występują dwa wyrazy pisane wielką literą (kandydat na imię i nazwisko), stara się połączyć z którymś posłem z tabeli posłów wraz z informacjami o nim, lub zanotowuje mu inną rolę jeśli nie jest to możliwe - 'rząd', 'prezydent', 'inni'.

In [ ]:
def annotate(text, mps_index):

    #uproszcone role w tym systemie 
    speaker_role = {
        'Marszałek': 'marszałek',
        'Wicemarszałek': 'marszałek',
        'Poseł': 'poseł',
        'Minister': 'rząd',
        'Sekretarz': 'rząd',
        'Podsekretarz': 'rząd',
        'Prezydent': 'prezydent',
        'Prezes Rady Ministrów': 'rząd',
        'Wiceprezes Rady Ministrów': 'rząd',
        'Rzecznik Prawo Obywatelskich': 'inni',
        'Prezes Narodowego Banku Polskiego': 'inni'
    }

    #dokonuje podmiany na każde znalezione dopasowanie
    def replace(match):
        #tekst przed nazwiskiem (look-behind context)
        before = match.group(1)
        #imię i nazwisko
        name = match.group(2)

        #szukamy roli ostatnich 10 słowach przed nazwiskiem (np. "Podsekretarz Stanu w Ministerstwie Rodziny, Pracy i Polityki Społecznej" to 9 słów, najdłuższa nazwa ministerstwa istniejącego w 2015) 
        role = None
        words = before.strip().split()[-10:]
        back_text = " ".join(words)
        #sprawdzamy czy któryś z tytułów ze słownika występuje przed nazwiskiem
        for key, value in speaker_role.items():
            if key in back_text:
                role = value
                break
        #jeśli rola to poseł to szukam w swoim spisie posłów informacji o nim
        if role == 'poseł' and name in mps_index.index:
            row = mps_index.loc[name]

            club = row.get("club", "brak")
            district = row.get("districtNum", "brak")

            return f"[{name}, poseł, {district}, {club}]:"
        #jeśli nie mamy więcej informacji 
        if role:
            return f"[{name}, {role}]:"


        return f"[{name}, inni]:"
    #szukam REGEX: kontekst do 10 słów, imię, nazwisko+dwukropek
    pattern = re.compile(
        r"((?:\b\w+\b\s+){0,10})"
        r"([A-ZĄĆĘŁŃÓŚŹŻ][a-ząćęłńóśźż]+\s+"
        r"[A-ZĄĆĘŁŃÓŚŹŻ][a-ząćęłńóśźż]+):"
    )
    #zwracam podmieniony tekst
    return pattern.sub(replace, text)


Tutaj wczytywane są informacje o posłach z odpowiedniego pliku dla żądanej kadencji Sejmu.

In [32]:
path_mps = os.path.join("mps", "kadencja_8.pkl")
mps = load_pickle_stream(path_mps)
mps = pd.DataFrame(mps)
out = (mps.iloc[0].to_list())

result = pd.json_normalize(out)
result

,accusativeName,active,birthDate,birthLocation,club,districtName,districtNum,educationLevel,firstLastName,firstName,genitiveName,id,inactiveCause,lastFirstName,lastName,numberOfVotes,profession,secondName,voivodeship,waiverDesc
0,Adama Abramowicza,False,1961-03-10,Biała Podlaska,PiS,Chełm,7,wyższe,Adam Abramowicz,Adam,Adama Abramowicza,1,Zrzeczenie,Abramowicz Adam,Abramowicz,10500,poseł na Sejm,Krzysztof,lubelskie,NaN
1,Andrzeja Adamczyka,False,1959-01-04,Krzeszowice,PiS,Kraków,13,wyższe,Andrzej Adamczyk,Andrzej,Andrzeja Adamczyka,2,NaN,Adamczyk Andrzej,Adamczyk,18514,parlamentarzysta,Mieczysław,małopolskie,NaN
2,Zbigniewa Ajchlera,False,1955-11-21,Szamotuły,PO-KO,Piła,38,wyższe,Zbigniew Ajchler,Zbigniew,Zbigniewa Ajchlera,3,NaN,Ajchler Zbigniew,Ajchler,7275,agroprzedsiębiorca,Czesław,wielkopolskie,NaN
3,Adama Andruszkiewicza,False,1990-06-30,Grajewo,niez.,Białystok,24,wyższe,Adam Andruszkiewicz,Adam,Adama Andruszkiewicza,4,NaN,Andruszkiewicz Adam,Andruszkiewicz,15668,specjalista ds. medialnych,NaN,podlaskie,NaN
4,Waldemara Andzela,False,1971-09-17,Czeladź,PiS,Sosnowiec,32,wyższe,Waldemar Andzel,Waldemar,Waldemara Andzela,5,NaN,Andzel Waldemar,Andzel,12021,poseł na Sejm RP,Franciszek,śląskie,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
500,Henryka Wnorowskiego,False,1963-02-05,Wnory - Stare,PiS,Białystok,24,wyższe,Henryk Wnorowski,Henryk,Henryka Wnorowskiego,501,NaN,Wnorowski Henryk,Wnorowski,6431,nauczyciel akademicki,Jan,podlaskie,NaN
501,Agnieszkę Soin,False,1977-02-09,Szczytno,PiS,Wrocław,3,wyższe,Agnieszka Soin,Agnieszka,Agnieszki Soin,502,NaN,Soin Agnieszka,Soin,3527,manager,Anna,dolnośląskie,NaN
502,Adama Kałaskę,False,1975-10-19,Ryki,PiS,Lublin,6,wyższe,Adam Kałaska,Adam,Adama Kałaski,503,NaN,Kałaska Adam,Kałaska,3388,przedsiębiorca prywatny,Krzysztof,lubelskie,NaN
503,Alicję Dąbrowską,False,1956-06-28,Ostrowiec Świętokrzyski,PO-KO,Warszawa,19,wyższe,Alicja Dąbrowska,Alicja,Alicji Dąbrowskiej,504,NaN,Dąbrowska Alicja,Dąbrowska,2483,lekarz,NaN,mazowieckie,NaN


Tutaj używam funkcji do anotacji tekstu informacjami o posłach. Iteruję po wszystkich stenogramach w danej kadencji.

In [33]:
mps_index = result.set_index("firstLastName")

annotated_texts = {}
for i, text in enumerate(texts):
    annotated_texts[i] = annotate(texts[i], mps_index)
#annotated_text = annotate(texts[5], mps_index)

Funkcje do parsowania całego tekstu. 

In [ ]:
#REGEXY: poszukuję wyrażeń, które mają strukturę [<mówca>]: <mowa> i szukam ich tak długo, dopóki nie trafię na kolejny nawias z [<mówca>]
SPEAKER_RE = re.compile(
    r"""
    \[
        (?P<speaker>[^\]]+)
    \]:
    \s*
    (?P<speech>.*?)
    (?=
        \n?\[.*?\]:
        |$
    )
    """,
    re.DOTALL | re.VERBOSE
)

#korzystam z powyższego wzorca, aby zrobić split na treść wystąpienia i metadane o nim, a następnie tworzę listę słowników 
def split_annotated_text(text: str):
    speeches = []

    for m in SPEAKER_RE.finditer(text):
        speaker_meta = m.group("speaker")
        speech = m.group("speech").strip()

        parts = [p.strip() for p in speaker_meta.split(",")]

        entry = {
            "name": parts[0],
            "role": parts[1] if len(parts) > 1 else None,
            "district": parts[2] if len(parts) > 2 else None,
            "club": parts[3] if len(parts) > 3 else None,
            "text": speech
        }

        speeches.append(entry)
    return speeches

#muszę czasem dodatkowo dopisać marszałka, ponieważ często jego wypowiedź była błędnie przypisywana do treści wystąpienia poprzedniego mówcy
def force_marshal_annotation(text: str) -> str:
    return re.sub(
        r"(?<!\[)\b(Marszałek|Wicemarszałek)\s*:",
        r"[\1, marszałek]:",
        text
    )


Stosuję powyższe funkcje żeby anotować tekst i podzielić go zgodnie z wypowiedziami. 

In [ ]:
'''
entries = {}

for text_index in annotated_texts:
    entry = annotated_texts[text_index]
    entry = force_marshal_annotation(entry)
    entry = split_annotated_text(entry)
    entries[text_index] = entry
'''

Ręcznie wypisałem imiona/nazwisko ze źle zidentyfikowanych mówców z kategorii 'inni', aby je wyeliminować - część informacji niestety zostanie utracona. Algorytm identyfikujący mówców jest dość naiwnie napisany, stąd dużo błędów. Eliminuję także wyrazy zakończone końcówkami fleksyjnymi, które wskazywałyby, że wyraz nie jest podany w formie w mianowniku - a tak nie są podawane w stenogramach przy faktycznych mówcach. 

In [ ]:
#wyrazy pisane wielką literą błędnie zidetyfikowane jako imię lub nazwisko mówcy
#niektóre podałem jako 2 słowa które błędnie wyszukano, a niektóre jako wyciętą część słowa bez końcówki, aby dopasowało do większej liczby słów
NON_PERSON_PATTERNS = [
    r"Platform",
    r"Prawo",
    r"Sprawiedliwo",
    r"Rodzin",
    r"Człowieka",
    r"Komisj",
    r"Rady",
    r"Polityki",
    r"Kaukazu",
    r"Konstytucj",
    r"Europ",
    r"Frug",
    r"Miast",
    r"Wolnoś",
    r"Niezawisł",
    r"Sejm",
    r"Podpowiad",
    r"Armii",
    r"Izb",
    r"Kontrol",
    r"Dni",
    r"Młodzie"
    r"Rzeczypospolit",
    r"Polskiej",
    r"Premier",
    r"Konkluduj",
    r"Kontrol",
    r"Wywiad",
    r"Prezydiu",
    r"Obron",
    r"Narodow",
    r"Telewizj",
    r"Thomas Jefferson", #ciekawe
    r"Prawic",
    r"Trybunal",
    r"Konstytuc",
    r"Józef Piłsudski" #ciekawe
    r"Służb",
    r"Celnej",
    r"Podsumowują",
    r"Rezerw",
    r"Materiałowyc",
    r"Wysokiej",
    r"Administracj",
    r"Skarbowe",
    r"Żeglug",
    r"Śródlądow",
    r"Dystrybucj",
    r"Polaków",
    r"Społecznyc",
    r"Jerzy Popiełuszko", #ciekawe
    r"Finansów",
    r"Publicznych",
    r"Przedsiębiorców",
    r"Pracodawców",
    r"Wschodu",
    r"Warmii Mazur",
    r"Abraham Lincoln", #ciekawe
    r"Związek Metropolitalny",
    r"Gospodarczej",
    r"Ubezpieczeń"
    r"Zespołu",
    r"Strażaków",
    r"Starszych Panów",
    r"Ireny Sendlerowej",
    r"Przykład Warszawy",
    r"Solidarności Walczącej",
    r"Marszu Niepodległości",
    r"Politechniki Śląskiej",
    r"Kanale Gliwickim",
    r"Wełny Mineralnej",
    r"Parlamentarnych Zespołów",
    r"Prokuratorii Generalnej",
    r"Cytat",
    r"Stabilnoś",
    r"Azotowych Puławy",
    r"Szanowni Państwo",
    r"Powtarzam",
    r"Inspekcji",
    r"Państw",
    r"Referencyjn",
    r"Wzorce",
    r"Sieci Badawczej",
    r"Teresy Kalkuty",
    r"Zabytków",
    r"Przypominam",
    r"Inicjatywy",
    r"Ustawodawcz",
    r"Podatkow",
    r"Analiz",
    r"Rzekłbym",
    r"Obrony Terytorialnej",
    r"Armii Krajowej",
    r"Port Komunikacyjny",
    r"Black Hawk",
    r"Danych Osobowych",
    r"Kończąc",
    r"Stronnictwo Ludowe",
    r"Diecezji Szczecińskiej",
    r"Wód Polskich",
    r"Poseł Suski",
    r"Fundusz",
    r"Sił Zbrojnych",
    r"Projektów Transportowych",
    r"Zjednoczonych Ameryki",
    r"Szanowny Przepraszam",
    r"Ordo Iuris",
    r"Funduszu Reprywatyzacji",
    r"Republik Radzieckich",
    r"Samorządow",
    r"Narodów",
    r"Zjednoczon",
    r"Pytam",
    r"Kogo Wiadomo",
    r"Wysokiej",
    r"Służb",
    r"Mówimy",
    r"Gospodarcz",
    r"Polsat News",
    r"Onkologii Kobiecej",
    r"Radiofonii Telewizji",
    r"Kontroli Państwowej",
    r"Analiz Sejmowych",
    r"Koalicji",
    r"Ewangelii Odpowiadam",
    r"Chrztu Polski",
    r"Wisławy Szymborskiej",
    r"Puszczy Białowieskiej",
    r"No Dyrektor",
    r"Ochrony Kopalń",
    r"Poseł Kwitek",
    r"Joanny Muchy",
    r"Franciszku Blachnickim",
    r"Józef Tischner",
    r"William Szekspir",
    r"Jerzym Waldorffem",
    r"Beaty Szydło",
    r"Sądem Najwyższym",
    r"Elżbiety Rafalskiej",
    r"Descarter Paryżu",
    r"Stalowej Woli",
    r"Rzeczyposolitej Mosińskiej",
    r"Polkom Polakom",
    r"Polski Ludowej",
    r'Trzy Plus',
    r"Sojuszu Pacyfiku",
    r"Doing Business",
    r"Elżbiety Sobeckiej",
    r"Ewy Kopacz",
    r"Drozd"
]

# końcówki fleksyjne, które jawnie wskazują, że wyraz nie jest w mianowniku
DECLINED_ENDINGS = (
    "ie", "ego", "emu", "ą", "ę", "owi", "ami", "ach"
)



Wykorzystuję powyżej zdefiniowane błędnie wzorce do stworzenia funkcji, które je będą stąd usuwać.

In [ ]:
#funkcja służąca do sprawdzenia, czy string wygląda jak prawdziwe imię i nazwisko

def looks_like_real_person(name):
    #splituję na osobne stringi
    parts = name.split()
    #imię moze zawierać 2 lub 3 człony (straciłem )
    if not (2 <= len(parts) <= 3):
        return False
    #sprawdzam czy pasuje do któregoś z listy błędnie zidentyfikowanych słów
    lname = name.lower()
    for p in NON_PERSON_PATTERNS:
        if re.search(p.lower(), lname):
            return False
    #sprawdzam czy pasuje do wzorca w którym każda część zaczyna się dużą literą
    for p in parts:
        if not re.match(r"[A-ZĄĆĘŁŃÓŚŹŻ][a-ząćęłńóśźż\-]+$", p):
            return False
        #eliminuję wyrazy nie w mianowniku
        if p.lower().endswith(DECLINED_ENDINGS):
            return False
    
    return True


def fix_false_speakers(entries):
    #lista na poprawną identyfikację osoby
    fixed = []
    #iteruję po każdej wypowiedzi
    for e in entries:
        name = e["name"].strip()
        lname = name.lower()
        #marszałek często w tekście występuje przed swoim wystąpieniem jako "Marszałek:" lub "Wicemarszałek:", wobec czego filtr 
        #looks_like_person by go błędnie zidentyfikowało, ta część go ratuje, lecz zrobiona naiwnie traci informację, który to był 
        #z marszałków
        if "marszałek" in lname:
            fixed.append({"name": "Marszałek","role": "marszałek","district": None,"club": None,"text": e["text"]})
            continue

        #jeśli nie wygląda na człowieka zgodnie z poprzednim wzorcem, to uznaję, że jest to część tekstu poprzedniego mówcy
        #(przykład: często identyfikowano "Wysoka Izbo" jako imię i nazwisko)
        if not looks_like_real_person(name):
            if fixed:
                fixed[-1]["text"] += " " + e["text"]
            continue

        fixed.append(e)

    return fixed


Naprawianie tekstu za pomocą powyższych funkcji.

In [ ]:
'''
all_entries = []

for doc_id, annotated in annotated_texts.items():
    entries = split_annotated(annotated)
    entries = fix_false_speakers(entries)

    for e in entries:
        e["doc_id"] = doc_id
        all_entries.append(e)
'''

Część w której faktycznie wszystko jest łączone w jedno i wszystkie teksty przetwarzane są w jednej pętli - dla uporządkowania.

In [ ]:
all_entries = []

# Iteracja po wyczyszczonych już tekstach
for doc_id, text in enumerate(texts):
    
    #adnotowanie mówców
    annotated_text = annotate(text, mps_index)
    
    #prawidłowa identyfikacja marszałka
    annotated_text = force_marshal_annotation(annotated_text)
    
    #rozbicie tekstu na listę wypowiedzi
    speeches = split_annotated_text(annotated_text)
    
    #naprawienie fałszywych mówców
    fixed_speeches = fix_false_speakers(speeches)
    
    #dodanie do listy ostatecznej wszystkich wypowiedzi dodając jednocześnie id dokumentu z którego wyciągnięto daną wypowiedź
    for entry in fixed_speeches:
        entry["doc_id"] = doc_id
        all_entries.append(entry)

#zapisanie w pandas.DataFrame dla wygody
df_results = pd.DataFrame(all_entries)
print(f"Przetworzono {len(df_results)} wypowiedzi.")



Przetworzono 56653 wypowiedzi.


Reszta kodu to sprawdzenie poprawności. 

In [61]:
all_entries_pd = df_results
all_entries_pd['text']  

0        Dziękuję. Dziękuję, pani poseł. Przystępujemy ...
1        Panie Marszałku Wysoka Izbo Pani Premier imien...
2        Proszę bardzo dziękuję panu posłowi. Jak słysz...
3        Panie Marszałku Wysoka Izbo Tak państwa to roz...
4        Proszę bardzo, tylko nie słyszałem pytania. Gł...
                               ...                        
56648    Pani Marszałkini Wysoka Izbo Pani Minister Ja ...
56649    Pani Marszałek Wysoka Izbo Pani Minister Bardz...
56650    Proszę państwa, zakończę takim razie kwestie m...
56651    Prostuję. Pismo Libert nie jest dotowane budże...
56652    Pani Marszałek Wysoka Izbo Mam zaszczyt przeds...
Name: text, Length: 56653, dtype: object

Zapisuję, aby móc użyć do analizy sentymentu w kolejnym notebooku. 

In [ ]:
all_entries_pd.to_csv('annotated_texts_8_term.csv')

Sprawdzam jak się wczytuje.

In [ ]:
import pandas as pd
all_entries = pd.read_csv('annotated_texts_8_term.csv')


,Unnamed: 0,name,role,district,club,text,doc_id
0,0,Marszałek,marszałek,NaN,NaN,"Dziękuję. Dziękuję, pani poseł. Przystępujemy ...",0
1,1,Marek Ast,poseł,8.0,PiS,Panie Marszałku Wysoka Izbo Pani Premier imien...,0
2,2,Marszałek,marszałek,NaN,NaN,Proszę bardzo dziękuję panu posłowi. Jak słysz...,0
3,3,Robert Kropiwnicki,poseł,1.0,PO-KO,Panie Marszałku Wysoka Izbo Tak państwa to roz...,0
4,4,Marszałek,marszałek,NaN,NaN,"Proszę bardzo, tylko nie słyszałem pytania. Gł...",0
...,...,...,...,...,...,...,...
56648,56648,Krzysztof Mieszkowski,poseł,3.0,PO-KO,Pani Marszałkini Wysoka Izbo Pani Minister Ja ...,234
56649,56649,Marcin Celiński,inni,NaN,NaN,Pani Marszałek Wysoka Izbo Pani Minister Bardz...,234
56650,56650,Marcin Celiński,inni,NaN,NaN,"Proszę państwa, zakończę takim razie kwestie m...",234
56651,56651,Marcin Celiński,inni,NaN,NaN,Prostuję. Pismo Libert nie jest dotowane budże...,234


Kogo znalazło jako osoby powiązane z rządem? Uwaga: w przypadku gdy osoba jest jednocześnie posłem i podsekretarzem/sekretarzem stanu w rządzie, to częściej jest identyfikowana jako należąca do rządu.

In [ ]:
all_entries[all_entries['role'] == 'rząd']


,Unnamed: 0,name,role,district,club,text,doc_id
83,83,Bogdan Święczkowski,rząd,NaN,NaN,Szanowny Panie Marszałku Szanowni Państwo Jest...,0
105,105,Bogdan Święczkowski,rząd,NaN,NaN,Szanowny Panie Marszałku Szanowni Państwo Chce...,0
192,192,Zbigniew Ziobro,rząd,NaN,NaN,Panie Marszałku Wysoka Izbo Mam okazję przekaz...,0
293,293,Andrzej Adamczyk,rząd,NaN,NaN,Dziękuję bardzo. Panie Marszałku Pani Premier ...,0
298,298,Witold Waszczykowski,rząd,NaN,NaN,Panie Prezydencie Panie Marszałku Pani Premier...,1
...,...,...,...,...,...,...,...
56608,56608,Bartosz Kownacki,rząd,NaN,NaN,Pani Marszałek Panie Pośle Chciałem tylko panu...,234
56616,56616,Witold Kołodziejski,rząd,NaN,NaN,Szanowna Pani Marszałek Szanowni Panie Posłank...,234
56617,56617,Witold Kołodziejski,rząd,NaN,NaN,"Dobrze. Jeśli chodzi tę kwestię, rzeczywiście ...",234
56619,56619,Konrad Szymański,rząd,NaN,NaN,Pani Marszałek Wysoka Izbo Mam przyjemność zas...,234
